In [1]:
import pandas as pd
import networkx as nx
from collections import defaultdict

In [2]:
relationships = pd.read_csv("data/hero_relationships.csv", keep_default_na=False).drop(columns=["Unnamed: 7", "Global Notes"])
hero_tier_list = pd.read_csv("data/hero_tier_list.csv").drop(columns=["Unnamed: 3"])
all_heroes = relationships["name"].to_list()
hero_tier_list[hero_tier_list["Name"] == "Frank"]

,Name,Position,Tier
50,Frank,Mid,A
98,Frank,Top,S


In [3]:
# Numeric map of tiers 
tier_map = {
    "S": 5,
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 1
}

hero_tier_list["tier_score"] = hero_tier_list["Tier"].map(tier_map)
heros = hero_tier_list["Name"].unique().tolist()
print(heros[:5])

['Ada', 'BaJie', 'Bariel', 'Bond', 'Bunu Shan']


In [4]:
def get_hero_best_positions(hero_name: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    names = [hero_name] if isinstance(hero_name, str) else hero_name.copy()

    for name in names:
        hero_rows = hero_tier_list[hero_tier_list["Name"] == name]
        max_score = hero_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_positions = hero_rows.loc[
            hero_rows["tier_score"] == target_score,
            ["Position", "Tier"]
        ].to_dict("records")

        if not best_positions:
            print(f"No heroes found under the name {name}.")
            continue

        position_names = [position["Position"] for position in best_positions]
        print_out = f"{name}'s best position is {', '.join(position_names)}."

        if verbose:
            print(print_out)

        output[name] = best_positions

    return output

#best_positions = get_hero_best_positions(["Ada", "Felyn", "Foso", "Frank"], verbose=True)
best_positions = get_hero_best_positions("Frank", verbose=True, tier=1)
print(best_positions)

Frank's best position is Top.
{'Frank': [{'Position': 'Top', 'Tier': 'S'}]}


In [5]:
# Get the best heroes for a position
def get_position_best_heroes(position: str | list[str], tier=1, verbose=False) -> dict[str, list[dict]]:
    output = {}

    positions = [position] if isinstance(position, str) else position.copy()

    for position in positions:
        position_rows = hero_tier_list[hero_tier_list["Position"] == position]
        max_score = position_rows["tier_score"].max()
        target_score = max_score - (tier - 1)

        best_heroes = position_rows.loc[
            position_rows["tier_score"] == target_score,
            ["Name", "Tier"]
        ].to_dict("records")

        if not best_heroes:
            print(f"No heroes found for the {position} position.")
            continue

        hero_names = [hero["Name"] for hero in best_heroes]
        print_out = f"Best heroes for the {position.lower()} position are {', '.join(hero_names)}."

        if verbose:
            print(print_out)

        output[position] = best_heroes

    return output

get_position_best_heroes(["Top", "Mid"], verbose=True)

Best heroes for the top position are Frank, Kid, Wukong, Foso.
Best heroes for the mid position are Aurelio, Merisi, Wolfgang, Foso.


{'Top': [{'Name': 'Frank', 'Tier': 'S'},
  {'Name': 'Kid', 'Tier': 'S'},
  {'Name': 'Wukong', 'Tier': 'S'},
  {'Name': 'Foso', 'Tier': 'S'}],
 'Mid': [{'Name': 'Aurelio', 'Tier': 'S'},
  {'Name': 'Merisi', 'Tier': 'S'},
  {'Name': 'Wolfgang', 'Tier': 'S'},
  {'Name': 'Foso', 'Tier': 'S'}]}

In [6]:
# Confirm heroes in relationships are == to heroes in tiers 
heroes_t = set(hero_tier_list["Name"].unique().tolist())
all_heroes = set(all_heroes)


mismatches = []
for hero in heroes_t:
    if hero in all_heroes:
        continue
    mismatches.append((hero))

if (len(heroes_t) != len(all_heroes)) or mismatches:
    print("Names do not match.")
else:
    print("Names match.")

Names match.


In [7]:
# Build tag -> hero lookup
lookup = defaultdict(list)
for hero in all_heroes:
    hero_data = relationships[relationships["name"] == hero]
    hero_tags = hero_data["tags"].tolist()[0].split(",")
    for tag in hero_tags:
        lookup[tag].append(hero)
print(lookup)

defaultdict(<class 'list'>, {'Round 1 Damage': ['Reinhardt', 'Felyn', 'Hakuna'], 'One Slap Chap': ['Reinhardt', 'Chen Fengcheng'], 'Initiator': ['Reinhardt', 'Hakuna', 'Aurelio'], 'Tank': ['Cubey', 'Merisi', 'Miki', 'Bart', 'Gillis', 'Peter', 'Matata', 'Fatty White', 'Frank', 'Big Foot'], 'Shield': ['Palulu', 'Miki', 'Qin Hu'], 'Multi-Attacker': ['Mo', 'Kamaitachi', 'Zealot', 'Elemi'], 'Fighter': ['Mo', 'Aurelio'], 'Debuff': ['Lady Deadfire', 'Felicity', 'Tiger Boy', 'Crank', 'Shougong Lei', 'Big Foot'], 'Lane Bully': ['Merisi', 'Qube', 'Felyn', 'Wolfgang'], 'Push': ['Gang', 'Felyn', 'Justice'], 'AoE': ['Gang', 'Foso'], 'Physical': ['Deep Space'], 'Ranged': ['Deep Space', 'Bunu Shan'], 'ADC': ['Deep Space', 'Omaha', 'Elemi', 'Ada', 'Bond', 'Quinn'], 'Search': ['Blocker', 'Beverly'], 'Gold Generation': ['Kid', 'BaJie', 'Acedia'], 'Assassin': ['Raven', 'Gillis', 'Xiangxi Ke', 'Lubos', 'Lan', 'Manta'], 'Crit': ['Raven', 'BaJie', 'Bariel', 'Quinn', 'Manta'], 'Skirmisher': ['Raven', 'Zealot

In [8]:
# Build hero tier and master lookups
import random
tiers = defaultdict(dict)
masteries = defaultdict(dict)
for _, row in hero_tier_list.iterrows():
    tiers[row["Name"]][row["Position"]] = row["Tier"]
    masteries[row["Name"]][row["Position"]] = random.randint(1, 7) # this is dyenamic in the final version

print(masteries["Frank"])
print(tiers["Frank"])


{'Mid': 6, 'Top': 3}
{'Mid': 'A', 'Top': 'S'}


In [9]:
# Make the graph
G = nx.DiGraph()

# Make a node for every hero
G.add_nodes_from(all_heroes)

# Add the position tier data
nx.set_node_attributes(G, tiers, name="tiers")
nx.set_node_attributes(G, masteries, name="masteries")

G.nodes["Aurelio"]

{'tiers': {'Jungler': 'S', 'Mid': 'S', 'Top': 'B'},
 'masteries': {'Jungler': 3, 'Mid': 6, 'Top': 7}}

In [10]:
# Resolve tags to heros
def resolve_targets(raw: str, lookup: dict, all_heroes: set) -> list[str]:
    if not raw or raw.strip().lower() == "none":
        return []
    targets = []
    for token in raw.split(","):
        token = token.strip()
        if token in all_heroes:
            targets.append(token)
        elif token in lookup:
            targets.extend(lookup[token])  # tag -> heroes
    return list(set(targets))  # deduplicate

In [11]:
# Edge columns
edge_types = {
    "synergies": "synergy",
    "counters": "counter",
    "countered_by": "countered_by",
    "anti-synergy": "anti_synergy",
}

for _, row in relationships.iterrows():
    hero = row["name"]
    for col, edge_type in edge_types.items():
        targets = resolve_targets(str(row[col]), lookup, all_heroes)
        for target in targets:
            if target == hero:
                continue
            G.add_edge(hero, target, type=edge_type)

In [35]:
G.out_edges("Kid", data=True)

OutEdgeDataView([('Kid', 'Xiangxi Ke', {'type': 'synergy'}), ('Kid', 'Raven', {'type': 'synergy'}), ('Kid', 'Manta', {'type': 'synergy'}), ('Kid', 'Mo', {'type': 'synergy'}), ('Kid', 'Lan', {'type': 'synergy'}), ('Kid', 'Gillis', {'type': 'countered_by'}), ('Kid', 'Lubos', {'type': 'synergy'}), ('Kid', 'Bart', {'type': 'countered_by'}), ('Kid', 'Qube', {'type': 'countered_by'}), ('Kid', 'Peter', {'type': 'countered_by'}), ('Kid', 'Miki', {'type': 'countered_by'}), ('Kid', 'Cubey', {'type': 'countered_by'}), ('Kid', 'Fatty White', {'type': 'countered_by'}), ('Kid', 'Big Foot', {'type': 'countered_by'}), ('Kid', 'Matata', {'type': 'countered_by'}), ('Kid', 'Hass', {'type': 'countered_by'}), ('Kid', 'Frank', {'type': 'countered_by'}), ('Kid', 'Merisi', {'type': 'countered_by'})])

In [12]:
# Build out a subgraph
def build_match_graph(G, team_a_available, team_b_available):
    all_available = set().union(*team_a_available.values(), *team_b_available.values())
    return G.subgraph(all_available).copy()

In [13]:
def flatten_available(available: dict) -> set:
    return set().union(*available.values())

def score_hero(
        G, 
        candidate,
        position, # for tier and mastery scoring
        t1_available: set, t1_picked: set, 
        t2_available: set, t2_picked: set
    ):
    
    # Team Comp Weights
    w_countered = 1.5
    w_synergy = 1
    w_counter = 1.5
    w_a_synergy = 0.5

    # Mastery weight
    w_mastery = 0.25

    # Tier Weight
    w_tier = 0.5

    # Init score
    score = 0

    # TODO Could be an opportunity here to split weights on picked vs could pick.  
    # TODO Could be an opportunity to also look at the masteries for the enemy heros (countering high masteries is more important)

    t1_available = flatten_available(t1_available)
    t2_available = flatten_available(t2_available)

    #1. Is there anything that counters this hero that can be picked against it?
    countered_by = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "countered_by"}
    for hero in countered_by:
        if hero in (t2_available | t2_picked):
            score -= w_countered
    
    # 2. What synergies are available for this hero
    synergies = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "synergy"}
    for hero in synergies:
        if hero in (t1_available | t1_picked):
            score += w_synergy

    # 3. Are there opportunities to counter enemy heroes?
    counters = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "counter"}
    for hero in counters:
        if hero in (t2_available | t2_picked):
            score += w_counter

    # 4. Are there any issues with picking this hero with our current heroes?
    anti_synergy = {v for _, v, d in G.out_edges(candidate, data=True) if d["type"] == "anti_synergy"}
    for hero in anti_synergy:
        if hero in (t1_available | t1_picked):
            score -= w_a_synergy

    # 5 - Add weight for tier
    score += tier_map[G.nodes[candidate]["tiers"][position]] # use the tier map to convert tier in this pos to numeric

    # 6 - Add weight for mastery
    score += G.nodes[candidate]["masteries"][position]

    return score

In [14]:
def build_position_availability(tier_data, team_available):
    position_availability = defaultdict(set)
    for _, row in tier_data.iterrows():
        hero = row["Name"]
        position = row["Position"]
        if hero in team_available:
            position_availability[position].add(hero)
    return position_availability

In [15]:
# Simulate a draft 
t1_roster = {"Frank", "Kid", "Wukong", "Hass", "Qube", "Cubey", "Reinhardt", "Merisi", "Foso", 
             "Zealot", "Wolfgang", "Aurelio", "Crank", "Manta", "Raven", "Dylan", "Kamaitachi", 
             "Niels", "Felyn", "BaJie", "Gang", "Deep Space", "Acedia", "Blocker", "Peiniang Zhu", 
             "Paisai", "Anna", "Peter", "Lady Deadfire"}

t2_roster = {"Frank", "Kid", "Wolfgang", "Hass", "Qube", "Mihawk", "Zealot", "Bart", "Aurelio", 
             "Manta", "Raven", "Lady Deadfire", "Lan", "Kamaitachi", "Leon", "Lubos", "Gillis", 
             "Bariel", "Felyn", "BaJie", "Quinn", "Ada", "Elemi", "Omaha", "Bond", "Acedia", 
             "Blocker", "Beverly", "Tivie", "Charon", "Felicity"}

t1_available = build_position_availability(hero_tier_list, t1_roster)
t2_available = build_position_availability(hero_tier_list, t2_roster)

# Who has already been picked?
t1_picked = set()
t2_picked = set()

# BUild out the match graph
G_match = build_match_graph(G, t1_available, t2_available)

In [16]:
def recommend_pick(G, lane, t1_available, t1_picked, t2_available, t2_picked):
    
    # Score all candidates for the requested lane
    candidates = t1_available[lane]
    scores = {
        hero: score_hero(G, hero, lane, t1_available, t1_picked, t2_available, t2_picked)
        for hero in candidates
    }
    
    # Get the best pick for this lane
    best = max(scores, key=scores.get)
    best_score = scores[best]
    
    # Check if best pick scores higher in another lane
    flag = None
    other_scores = {
        other_lane: score_hero(G, best, other_lane, t1_available, t1_picked, t2_available, t2_picked)
        for other_lane, pool in t1_available.items()
        if other_lane != lane and best in pool
    }
    if other_scores:
        best_alt_lane = max(other_scores, key=other_scores.get)
        if other_scores[best_alt_lane] > best_score:
            flag = f"Consider {best} for {best_alt_lane} instead (scores {other_scores[best_alt_lane]:.1f} vs {best_score:.1f} here)"
    
    return best, best_score, flag, scores


# Usage
best, score, flag, all_scores = recommend_pick(G_match, "Jungler", t1_available, t1_picked, t2_available, t2_picked)
print(f"Recommended: {best} ({score})")
print(f"All scores: {sorted(all_scores.items(), key=lambda x: x[1], reverse=True)}")
if flag:
    print(f"WARNING: {flag}")

Recommended: Reinhardt (16)
All scores: [('Reinhardt', 16), ('Kamaitachi', 12.0), ('Manta', 9.5), ('Zealot', 9), ('Aurelio', 6.0), ('Raven', -2.5)]


In [62]:
# Hero set manipulation functions
def pick_hero(t_picked: set, hero: str):
    for position in t1_available.values():
        position.discard(hero)
    for position in t2_available.values():
        position.discard(hero)
    t_picked.add(hero)

def ban_hero(hero: str):
    for position in t1_available.values():
        position.discard(hero)
    for position in t2_available.values():
        position.discard(hero)

In [83]:
ban_hero("Frank")

In [87]:
pick_hero(t1_picked, "Aurelio")

In [91]:
pick_hero(t2_picked, "Cubey")